# E7 step 2 · The whole chain on the unseen test pairs (Kaggle T4)

Task **E7**, step 2, from `Essential_task_delegation/Next_Steps_Roadmap.md` §1b. Built 7 Nov 2026. It trains nothing.

**The chain being scored**
1. **Find:** SAM 3 (a text-prompted segmentation model, prompt "building") outlines each building on the before-photo, cut into four quarters. Settings as in E3 and R16c step 4.
2. **Grade:** a Siamese ResNet-18 reads 128 px before and after crops of one outline and labels it intact, damaged, destroyed or not a building. It was trained on SAM 3's outlines of the training images (R16c steps 5 and 5b). Three graders exist: seeds 42, 43 and 44. Each keeps the label biases picked on held-out training images.
3. **Table:** one row per building in H2's columns: 8-connected blobs of one class, at least 20 px, from grader seed 42's map. Seed 42 was fixed on 4 Nov, from its best clean validation score (0.3161), before any test score existed.

**The baseline:** R11, a U-Net with a ResNet-50 encoder that takes the after-photo and before-photo as 6 input channels and predicts all four classes, with 8-view test-time augmentation (TTA).

**Why**
- Our best numbers came from three different models and never from one chain on one set of images. This scores one chain.
- Every model so far was chosen on validation pairs. E6 (7 Nov) built masks for 2,007 clean pairs no segmentation model has seen.

**What it does**
- Runs SAM 3, the three graders and R11 on the 2,065 readable optical test pairs. Two of E6's 2,067 pairs have an unreadable image.
- **Map scores:** mean IoU and destroyed-building IoU on E6's 2,007 clean pairs, per grader and as the mean of three, per event, and with and without Turkey. Macro F1 of each grader on the true buildings, for BDAChat's comparison.
- **Count scores:** the fixed query and the nearest option, as in H2, on the optical counting questions, all and without the 66 whose after-image E5 found copied (E5's clean ones), against R11's table on the same questions (exact McNemar) and the 34.2% bar.
- **Added 7 Nov, before any test score exists:** count error against each question's true count, and the rule that never sees the image (`benchmark-option-rule-check.ipynb`: 66.98% on these questions). They are reported beside accuracy and don't decide anything. Why: accuracy on these options can't show that a method counts well, because the middle option is always right when the largest option is above 20. E6's masks give each question's exact true count on all 3,470 questions on the clean pairs (checked on the PC, 7 Nov).

**Rule, fixed before the run (4 Nov)**
- The chain stays SAM 3 + grader if its destroyed-building IoU (mean of three graders) is above R11's on the clean test pairs, and its counting isn't clearly below R11's (p < 0.05).
- If its counting is clearly below but its destroyed score is above, both results go to Abrar; there's no automatic choice.
- If its destroyed score isn't above R11's, the lead doesn't hold on unseen data, and the choice of chain reopens before T2 step 5.
- O1 stays judged on validation, as the review set it. The test score is reported beside it, and nothing is ever tuned on the test pairs.
- **How "clearly below" is read (fixed 7 Nov, before the run):** seed 42's table against R11's on E5's clean questions, the chain's accuracy lower, exact McNemar p < 0.05. All the optical questions are reported too.

**Expectation, fixed before the run**
- Every model scores lower on test than on validation (clean validation: chain 0.3065 mean of three, R11 with TTA 0.2692): Turkey is 20% of the test pairs against 10%, five events are new to validation (Mexico, Joplin, Ian, Noto, Guatemala), and no twins remain. So the chain may fall below 0.30 here. No test number is predicted.
- Counting about level with R11's. On validation (step 1, 7 Nov) the chain answers 37.73% against R11's 39.24%, p = 0.39, with median count error 40% against 33%.

**Safety checks before the long run** (Cell 7)
- SAM 3 here must reproduce E3's saved maps on validation images.
- Each grader must reproduce its saved log-probabilities on those images, and R11 its saved pixels. If any fails, the notebook stops before the long run.
- Afterwards, R11's table must reproduce H2's 32.96% on the same questions.

**How to run on Kaggle** (account 1, whose Hugging Face token has SAM 3 access)
1. Settings: GPU T4 x2 (one is used), internet ON, secret `HF_TOKEN` (AbrarAlam: approved for `facebook/sam3`, and R11's checkpoint is private).
2. Inputs:
   - `abrarmohammedtanzim/disasterm3-mirror`;
   - `redwannewaz/disasterm3-bench-mirror-v2`;
   - the private dataset `e7-inputs`, made from `Champion_Task/E7-inputs/e7-inputs.zip` (E6's `test_pairs.csv` and masks, E5's `bench_questions_flagged.csv`);
   - Your Work → `e3-sam3-building-finding`;
   - Your Work → `r16c-step5-grader-on-sam3-outlines`;
   - Your Work → `r16c-step5b-third-grader-seed`.
3. Run All once with `SMOKE_TEST = True` in Cell 2 (12 test pairs, 3 validation pairs). If Cell 1 says to restart the kernel, restart and Run All again. Then set it to `False` and use Save & Run All.
   - Time, estimated from E3 and R16c step 5: SAM 3 about 75 min, then the graders and R11 about 2 h, so about 3.5 h. No time is promised: this exact chain hasn't run on this many images.
4. Outputs in `/kaggle/working/step2/`:
   - `README.md`, `config.json`;
   - `map_scores.csv` (clean pairs, with and without Turkey), `map_scores_by_event.csv`, `grader_true_buildings.csv`;
   - `counting_scores.csv`, `per_question.csv`;
   - `instances_test.csv`: each found building with its three graders' log-probabilities, R11's pixels and, on clean pairs, the true pixels;
   - `buildings_chain_s42.sqlite`: the table;
   - `chain_s42_maps.zip`, `r11_tta_maps.zip`, `sam3_pre_grid2.zip`: for step 3 (hot spots, on the PC).


In [1]:
# ── Cell 1: Environment (Run-All safe): R11's library here, SAM 3 in its own folder ──
# segmentation-models-pytorch 0.3.4 is needed to load R11 (its checkpoint key layout), and it pins timm 0.9.7. SAM 3 needs
# timm 1.0.17 or later. So SAM 3 and its packages go in their own folder, and Cell 5 runs SAM 3 in a separate process that
# reads that folder first. SAM 3 is installed without its dependency list, as in E3: it pins numpy<2, which has no wheel for
# Kaggle's Python 3.13.
import importlib.metadata as _md
import os, sys, subprocess

PINS = {
    "segmentation-models-pytorch": "0.3.4",   # baseline version (checkpoint key layout)
}

def _mismatched():
    out = []
    for pkg, want in PINS.items():
        try:
            have = _md.version(pkg)
        except _md.PackageNotFoundError:
            have = "not installed"
        if have != want:
            out.append(f"{pkg}: {have} → {want}")
    return out

if _mismatched():
    print("⏳ Installing pinned versions:")
    for _m in _mismatched():
        print(f"   {_m}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *(f"{k}=={v}" for k, v in PINS.items()), "huggingface_hub[hf_xet]"],
                   check=True)                # pip failure stops the run here
    if _mismatched():
        raise RuntimeError(f"Pins still wrong after install: {_mismatched()}")
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
        # background "Save & Run All": nothing imported yet, so new versions load on first import
        print("✓ Dependencies installed (background run) — continuing without restart.")
    else:
        raise SystemExit("✓ Dependencies installed. RESTART THE KERNEL NOW "
                         "(Run → Restart & clear cell outputs), then Run All again.")
else:
    print("✓ All pinned versions already installed — proceeding.")

SAM3_COMMIT = "7f473cfd983d5eb43040ff608f66bff0d7bdcb53"     # facebookresearch/sam3 main, 14 Oct 2026, as in E3
SAM_ENV = "/kaggle/temp/sam3_env"

def _pip_sam(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", SAM_ENV, *args], check=True)

if not os.path.isdir(os.path.join(SAM_ENV, "sam3")):
    os.makedirs(SAM_ENV, exist_ok=True)
    _pip_sam(f"git+https://github.com/facebookresearch/sam3.git@{SAM3_COMMIT}")
    # E3's list of what SAM 3 really imports at load time, plus the two packages iopath and ftfy need
    _pip_sam("timm>=1.0.17", "ftfy==6.1.1", "wcwidth", "regex", "iopath", "portalocker", "einops", "pycocotools", "psutil",
             "setuptools<81")

import torch
print(f"✓ SAM 3 in {SAM_ENV} (commit {SAM3_COMMIT[:7]}); torch {torch.__version__}, python {sys.version.split()[0]}")
print(f"  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (this notebook needs a GPU)'}")


⏳ Installing pinned versions:
   segmentation-models-pytorch: not installed → 0.3.4
✓ Dependencies installed (background run) — continuing without restart.
✓ SAM 3 in /kaggle/temp/sam3_env (commit 7f473cf); torch 2.11.0+cu130, python 3.13.16
  GPU: Tesla T4


In [2]:
# ── Cell 2: Configuration and inputs ──────────────────────────────────────
import ast
import csv
import json
import re
import shutil
import sqlite3
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from scipy import ndimage
from scipy.spatial import cKDTree
from scipy.stats import binomtest
from tqdm.auto import tqdm

SMOKE_TEST = False                  # True = 12 test pairs, 3 validation pairs for the safety checks
INPUT_ROOT = Path("/kaggle/input")
DATA_ROOT = INPUT_ROOT / "datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct"
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/step2")
MAP_DIR = Path("/kaggle/temp/sam3_pre_grid2")      # zipped into the output at the end
CHECK_DIR = Path("/kaggle/temp/sam3_check")
CHAIN_DIR, R11_DIR = Path("/kaggle/temp/chain_s42_maps"), Path("/kaggle/temp/r11_tta_maps")

NUM_CLASSES = 4                     # 0=Background, 1=Intact, 2=Damaged, 3=Destroyed
CLASS_NAMES = ["Background", "Intact", "Damaged", "Destroyed"]
GRADES = CLASS_NAMES[1:]
LABELS = GRADES + ["None"]          # the graders' 4th label: an outline that is mostly background
SEEDS = (42, 43, 44)                # R16c steps 5 and 5b
TABLE_SEED = 42                     # fixed 4 Nov, from its best clean validation score (0.3161), before any test score
CROP, MARGIN = 128, 32              # as the graders trained
MIN_AREA = 20                       # R16's smallest building piece, also H2's table rule
PROMPT, CONFIDENCE, GRID, MIN_INSTANCE_PX, PRECISION = "building", 0.5, 2, 10, "fp16"   # E3's and R16c step 4's values
CHECK_IMAGES = 12                   # validation images for the safety checks
UNET, HF_USER = "R11", "AbrarAlam"  # checkpoint: {HF_USER}/disasterm3-d4-R11/final_model.pth
TASK = "Building Damage Counting"
CLASS_OF = {"Intact building counting.": 1, "Damaged building counting.": 2, "Totally-destroyed building counting.": 3}
CLASS_NAME = {1: "intact", 2: "damaged", 3: "destroyed"}

BIAS_FIXED = {42: [0.0, -1.75, -4.0, -1.25], 43: [0.0, -1.0, -4.0, -2.0], 44: [0.0, -2.0, -4.0, -3.5]}   # calibration.json
EXPECTED = {"pairs": 2067, "readable": 2065, "clean": 2007, "questions": 3583, "e5_copied": 66, "rule_%": 66.98,
            "middle_%": 53.95, "r11_all_%": 32.96, "r11_clean_%": 32.47, "base_%": 34.2,
            "val_chain_destroyed": 0.3065, "val_r11_destroyed": 0.2692}
LIMITS = {"sam_iou": 0.97, "logp": 0.05, "r11_px": 0.02, "r11_points": 0.3}   # safety checks, Cells 7 and 10

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

TOKEN = hf_token()
if not TOKEN:
    raise RuntimeError("Add the HF_TOKEN secret: facebook/sam3 is gated, and R11's checkpoint is private.")

# One walk over the inputs: the files and folders this notebook looks for
WANTED = {"test_pairs.csv", "test_masks_4class.zip", "bench_questions_flagged.csv", "benchmark_release.json",
          "calibration.json", "instances_val.csv", *(f"grader_seed{s}.pth" for s in SEEDS)}
FOUND, FOUND_DIRS, PTH = defaultdict(list), defaultdict(list), []
for root, dirs, files in os.walk(INPUT_ROOT):
    for name in files:
        if name in WANTED:
            FOUND[name].append(Path(root) / name)
        if name.endswith(".pth"):
            PTH.append(Path(root) / name)
    for name in dirs:
        if name in ("pre_grid2", "test_masks_4class"):
            FOUND_DIRS[name].append(Path(root) / name)

def found(name, why):
    if not FOUND[name]:
        raise FileNotFoundError(f"No {name} under {INPUT_ROOT}: {why}")
    return sorted(FOUND[name])[0]

def find_checkpoint(name):
    """As in E1: a /kaggle/input folder named R11 / d4_R11 / R11-results, then Hugging Face."""
    wanted = {name.lower(), f"d4_{name}".lower(), f"{name}-results".lower()}
    hits = sorted((p for p in PTH if any(part.lower() in wanted for part in p.parts)), key=lambda p: p.name != "final_model.pth")
    if hits:
        return hits[0]
    return hf_hub_download(f"{HF_USER}/disasterm3-d4-{name}", "final_model.pth", token=TOKEN)

STEP5_WHY = "attach Your Work → r16c-step5-grader-on-sam3-outlines and r16c-step5b-third-grader-seed"
E7_INPUTS = "attach the private dataset e7-inputs, made from Champion_Task/E7-inputs/e7-inputs.zip"
GRADER_PATHS = {s: found(f"grader_seed{s}.pth", STEP5_WHY) for s in SEEDS}
TEST_PAIRS = found("test_pairs.csv", E7_INPUTS)
FLAGGED = found("bench_questions_flagged.csv", E7_INPUTS)
_bench = sorted(p for p in FOUND["benchmark_release.json"] if (p.parent / "test_images").is_dir())
if not _bench:
    raise FileNotFoundError("Attach redwannewaz/disasterm3-bench-mirror-v2 (benchmark_release.json next to test_images/)")
BENCH_JSON = _bench[0]
BENCH_IMAGES = BENCH_JSON.parent / "test_images"
if not FOUND_DIRS["pre_grid2"]:
    raise FileNotFoundError("No pre_grid2 folder: attach Your Work → e3-sam3-building-finding")
E3_DIR = sorted(FOUND_DIRS["pre_grid2"])[0]
# Kaggle may keep the masks' zip as a zip or unpack it
MASK_ZIP = zipfile.ZipFile(sorted(FOUND["test_masks_4class.zip"])[0]) if FOUND["test_masks_4class.zip"] else None
MASK_FOLDER = sorted(FOUND_DIRS["test_masks_4class"])[0] if FOUND_DIRS["test_masks_4class"] else None
if MASK_ZIP is None and MASK_FOLDER is None:
    raise FileNotFoundError(f"E6's merged masks are missing: {E7_INPUTS}")
assert MANIFEST_PATH.exists(), f"No manifest at {MANIFEST_PATH}: attach abrarmohammedtanzim/disasterm3-mirror"

BIASES = {}
for path in FOUND["calibration.json"]:
    for seed, chosen in json.loads(path.read_text(encoding="utf-8")).items():
        if int(seed) in SEEDS:
            BIASES[int(seed)] = np.array([chosen["bias"][l] for l in LABELS])
for seed in SEEDS:
    assert seed in BIASES and BIASES[seed].tolist() == BIAS_FIXED[seed], f"Seed {seed}'s biases aren't R16c's: {STEP5_WHY}"
INST_PATHS = {}
for path in FOUND["instances_val.csv"]:
    columns = pd.read_csv(path, nrows=0).columns
    for seed in SEEDS:
        if f"logp_intact_s{seed}" in columns:
            INST_PATHS[seed] = path
assert len(INST_PATHS) == len(SEEDS), f"instances_val.csv found for seeds {sorted(INST_PATHS)} only: {STEP5_WHY}"

UNET_CKPT = find_checkpoint(UNET)
for d in (MAP_DIR, CHECK_DIR, CHAIN_DIR, R11_DIR):
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ E7 step 2{'  [SMOKE TEST]' if SMOKE_TEST else ''}: graders {list(SEEDS)}, table from seed {TABLE_SEED}; U-Net {UNET_CKPT}")
print(f"  E6: {TEST_PAIRS}; masks in {'a zip' if MASK_ZIP else MASK_FOLDER}; benchmark: {BENCH_JSON.parent}; E3's maps: {E3_DIR}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}; outputs: {OUT_DIR}")


final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

✓ E7 step 2: graders [42, 43, 44], table from seed 42; U-Net /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-d4-R11/snapshots/31974dacfee2125a66639858433a113704edf9b3/final_model.pth
  E6: /kaggle/input/datasets/abrarmohammedtanzim/e7-inputs/test_pairs.csv; masks in /kaggle/input/datasets/abrarmohammedtanzim/e7-inputs/test_masks_4class; benchmark: /kaggle/input/datasets/redwannewaz/disasterm3-bench-mirror-v2; E3's maps: /kaggle/input/notebooks/abrarmohammedtanzim/e3-sam3-building-finding/E3/masks/pre_grid2
  Device: Tesla T4; outputs: /kaggle/working/step2


In [3]:
# ── Cell 3: Helpers (pure functions), with self-checks ──────────────────────
def name_of(path):
    return Path(str(path).replace("\\", "/")).name

XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")

def event_of(name):
    """E1's rule: 'hurricane_florence_00000261_post_disaster.png' -> 'hurricane_florence'."""
    m = XBD_NAME.match(name)
    if m:
        return m.group(1)
    m = OTHER_NAME.match(name)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem)

def read_rgb(path):
    image = cv2.imread(str(path))
    return None if image is None else cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

def read_pair(post_path, pre_path):
    """After-image and before-image (RGB, same size), as in the D4 dataset class; None if one can't be read."""
    post, pre = read_rgb(post_path), read_rgb(pre_path)
    if post is None or pre is None:
        return None
    if pre.shape != post.shape:
        pre = cv2.resize(pre, (post.shape[1], post.shape[0]), interpolation=cv2.INTER_LINEAR)
    return post, pre

def read_mask(name, shape):
    """E6's merged mask (0-3) at the image's size."""
    if MASK_ZIP is not None:
        mask = cv2.imdecode(np.frombuffer(MASK_ZIP.read(name), np.uint8), cv2.IMREAD_GRAYSCALE)
    else:
        mask = cv2.imread(str(MASK_FOLDER / name), cv2.IMREAD_GRAYSCALE)
    if mask.shape != shape:
        mask = cv2.resize(mask, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask

def read_sam(path, shape):
    """A SAM 3 instance map (16-bit PNG, 0 = nothing) at the image's size; zeros if it is missing."""
    sam = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if sam is None:
        return np.zeros(shape, np.int32)
    if sam.shape != shape:
        sam = cv2.resize(sam, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
    return sam.astype(np.int32)

def square_box(sl, h, w):
    """Square window around a building's (row, column) slices, MARGIN px wider on each side, kept inside the image."""
    y0, y1, x0, x1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
    side = min(max(y1 - y0, x1 - x0) + 2 * MARGIN, h, w)
    cy, cx = (y0 + y1) / 2, (x0 + x1) / 2
    top = int(round(min(max(cy - side / 2, 0), h - side)))
    left = int(round(min(max(cx - side / 2, 0), w - side)))
    return top, left, side

def building_crops(pre, post, labels, ids):
    """labels: int map of building ids (0 = none). Returns uint8 (len(ids), 7, CROP, CROP):
    before RGB, after RGB, and 255 where the pixel belongs to that building. R16c step 5's code."""
    h, w = labels.shape
    slices = ndimage.find_objects(labels)
    out = np.empty((len(ids), 7, CROP, CROP), np.uint8)
    for n, lid in enumerate(ids):
        top, left, side = square_box(slices[lid - 1], h, w)
        win = (slice(top, top + side), slice(left, left + side))
        out[n, 0:3] = cv2.resize(pre[win], (CROP, CROP), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
        out[n, 3:6] = cv2.resize(post[win], (CROP, CROP), interpolation=cv2.INTER_LINEAR).transpose(2, 0, 1)
        out[n, 6] = cv2.resize((labels[win] == lid).astype(np.uint8) * 255, (CROP, CROP),
                               interpolation=cv2.INTER_NEAREST)
    return out

def true_pieces(gt):
    """The true buildings of one image: the 8-connected pieces of each class, at least MIN_AREA px (R16's rule)."""
    out = []
    for cls in (1, 2, 3):
        n, labels, stats, _ = cv2.connectedComponentsWithStats((gt == cls).astype(np.uint8), connectivity=8)
        ids = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= MIN_AREA]
        if ids:
            out.append((cls - 1, labels, ids))
    return out

def to_class(labels):
    """Grader labels -> mask classes: intact, damaged, destroyed -> 1-3; "None" -> 0 (background)."""
    return np.where(labels < 3, labels + 1, 0)

# H2's table: the same columns, the same blobs (8-connected, one class, at least MIN_AREA px)
COLUMNS = [("image", "TEXT"), ("building_id", "INTEGER"), ("class_id", "INTEGER"), ("class_name", "TEXT"), ("area_px", "INTEGER"),
           ("centroid_x", "REAL"), ("centroid_y", "REAL"), ("bbox_x", "INTEGER"), ("bbox_y", "INTEGER"), ("bbox_w", "INTEGER"),
           ("bbox_h", "INTEGER"), ("dist_to_nearest_destroyed_px", "REAL")]

def create_schema(conn):
    cols = ",\n  ".join(f"{n} {t}" for n, t in COLUMNS)
    conn.execute(f"CREATE TABLE buildings (\n  {cols},\n  PRIMARY KEY (image, building_id))")
    conn.execute("CREATE TABLE images (image TEXT PRIMARY KEY, width INTEGER, height INTEGER)")
    conn.execute("CREATE INDEX idx_buildings_class ON buildings (image, class_name)")

def add_nearest_destroyed(blobs):
    for b in blobs:
        b["dist_to_nearest_destroyed_px"] = None
    dest = [b for b in blobs if b["class_id"] == 3]
    if not dest:
        return
    tree = cKDTree(np.array([[b["centroid_x"], b["centroid_y"]] for b in dest]))
    for b in blobs:
        p = (b["centroid_x"], b["centroid_y"])
        if b["class_id"] == 3:
            if len(dest) < 2:
                continue
            d = float(tree.query(p, k=2)[0][1])      # the nearest hit is the blob itself, at distance 0
        else:
            d = float(tree.query(p, k=1)[0])
        b["dist_to_nearest_destroyed_px"] = round(d, 1)

def blobs_of(pred):
    """Building rows for one 0-3 mask: 8-connected blobs per class, minus blobs under MIN_AREA."""
    out = []
    for cls, cname in CLASS_NAME.items():
        n, _, stats, cents = cv2.connectedComponentsWithStats((pred == cls).astype(np.uint8), connectivity=8)
        for k in range(1, n):
            area = int(stats[k, cv2.CC_STAT_AREA])
            if area < MIN_AREA:
                continue
            out.append({"class_id": cls, "class_name": cname, "area_px": area,
                        "centroid_x": round(float(cents[k][0]), 1), "centroid_y": round(float(cents[k][1]), 1),
                        "bbox_x": int(stats[k, cv2.CC_STAT_LEFT]), "bbox_y": int(stats[k, cv2.CC_STAT_TOP]),
                        "bbox_w": int(stats[k, cv2.CC_STAT_WIDTH]), "bbox_h": int(stats[k, cv2.CC_STAT_HEIGHT])})
    for i, b in enumerate(out, 1):
        b["building_id"] = i
    add_nearest_destroyed(out)
    return out

def insert_image(conn, name, height, width, blobs):
    conn.execute("INSERT INTO images VALUES (?, ?, ?)", (name, width, height))
    names = [n for n, _ in COLUMNS]
    conn.executemany(f"INSERT INTO buildings ({', '.join(names)}) VALUES ({', '.join('?' * len(names))})",
                     [tuple(name if n == "image" else b[n] for n in names) for b in blobs])

def blob_counts(pred):
    """[intact, damaged, destroyed]: blobs of at least MIN_AREA px, as the table counts them."""
    out = []
    for cls in (1, 2, 3):
        n, _, stats, _ = cv2.connectedComponentsWithStats((pred == cls).astype(np.uint8), connectivity=8)
        out.append(int((stats[1:n, cv2.CC_STAT_AREA] >= MIN_AREA).sum()))
    return out

def options_of(e):
    o = e["options_list"]
    return [float(str(x).rstrip("%")) for x in (ast.literal_eval(o) if isinstance(o, str) else o)]

def nearest_option(n, options):
    return "ABCDE"[min(range(len(options)), key=lambda i: abs(options[i] - n))]

def no_image_rule(options):
    """Never sees the image: largest option above 20 -> the middle value, otherwise the second smallest."""
    s = sorted(options)
    return "ABCDE"[options.index(s[2] if s[4] > 20 else s[1])]

def middle_option(options):
    return "ABCDE"[options.index(sorted(options)[2])]

def confusion(gt, pred):
    """4×4 pixel counts, rows = truth, columns = prediction."""
    idx = gt.astype(np.int64).ravel() * NUM_CLASSES + pred.astype(np.int64).ravel()
    return np.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)

def metrics(cm):
    """Dataset-level metrics from a confusion matrix (E1's formulas), plus building IoU (classes 1-3 merged)."""
    cm = cm.astype(np.float64)
    eps = 1e-8
    tp = np.diag(cm)
    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
    iou = tp / (tp + fp + fn + eps)
    precision, recall = tp / (tp + fp + eps), tp / (tp + fn + eps)
    building_tp = cm[1:, 1:].sum()
    building_iou = building_tp / (building_tp + cm[0, 1:].sum() + cm[1:, 0].sum() + eps)
    return {"miou": round(float(iou.mean()), 4), **{f"{c.lower()}_iou": round(float(iou[i]), 4) for i, c in enumerate(CLASS_NAMES)},
            "building_iou": round(float(building_iou), 4), "destroyed_precision": round(float(precision[3]), 4),
            "destroyed_recall": round(float(recall[3]), 4)}

def mcnemar(x, y):
    """Exact McNemar on two boolean series: only the questions one method gets right and the other wrong count."""
    only_x, only_y = int((x & ~y).sum()), int((~x & y).sum())
    p = binomtest(only_x, only_x + only_y, 0.5).pvalue if only_x + only_y else 1.0
    return {"only_chain_right": only_x, "only_R11_right": only_y, "p": round(float(p), 4)}

def verdict(chain_destroyed, r11_destroyed, chain_acc, r11_acc, p):
    """The rule fixed before the run. Returns (outcome, sentence)."""
    lead = chain_destroyed > r11_destroyed
    clearly_below = chain_acc < r11_acc and p < 0.05
    if not lead:
        return "reopen", ("The chain's destroyed-building IoU isn't above R11's: the lead doesn't hold on unseen data, and the "
                          "choice of chain reopens before T2 step 5.")
    if clearly_below:
        return "abrar", ("The chain's destroyed-building IoU is above R11's but its counting is clearly below: both results go "
                         "to Abrar, with no automatic choice.")
    return "stays", "The chain stays SAM 3 + grader: its destroyed-building IoU is above R11's and its counting isn't clearly below."

# Self-checks
_m = np.zeros((200, 200), np.uint8)
_m[10:20, 10:20] = 1                    # intact, 100 px
_m[20, 20] = 1                          # touches that blob on a diagonal: still one blob
_m[100:102, 100:102] = 1                # intact, 4 px: under MIN_AREA
_m[50:60, 50:60] = 3                    # destroyed
_m[50:60, 150:160] = 3                  # destroyed
_m[150:153, 150:153] = 3                # destroyed, 9 px: under MIN_AREA
_b = blobs_of(_m)
assert [b["class_name"] for b in _b].count("intact") == 1 and sum(b["class_id"] == 3 for b in _b) == 2
assert [b["dist_to_nearest_destroyed_px"] for b in _b if b["class_id"] == 3] == [100.0, 100.0]
assert blob_counts(_m) == [1, 0, 2] and len(_b) == sum(blob_counts(_m))
assert nearest_option(52, [10, 40, 60, 80, 100]) == "C" and no_image_rule([231, 323, 277, 139, 185]) == "A"
assert middle_option([3, 9, 4, 13, 19]) == "B" and no_image_rule([3, 9, 4, 13, 19]) == "C"
assert to_class(np.array([0, 1, 2, 3])).tolist() == [1, 2, 3, 0]
_g = np.array([[0, 1], [2, 3]], np.uint8)
assert metrics(confusion(_g, _g))["miou"] == 1.0
_x, _y = pd.Series([True, True, False, False]), pd.Series([True, False, True, False])
assert mcnemar(_x, _y) == {"only_chain_right": 1, "only_R11_right": 1, "p": 1.0}
assert verdict(0.3, 0.2, 40, 39, 0.5)[0] == "stays" and verdict(0.3, 0.2, 30, 39, 0.01)[0] == "abrar"
assert verdict(0.3, 0.2, 39, 30, 0.01)[0] == "stays" and verdict(0.2, 0.3, 30, 39, 0.01)[0] == "reopen"
_lab = np.zeros((200, 300), np.int32)
_lab[0:10, 0:20], _lab[100:150, 200:260] = 1, 2
assert square_box(ndimage.find_objects(_lab)[0], 200, 300) == (0, 0, 20 + 2 * MARGIN)
_c = building_crops(np.zeros((200, 300, 3), np.uint8), np.zeros((200, 300, 3), np.uint8), _lab, [1, 2])
assert _c.shape == (2, 7, CROP, CROP) and _c[1, 6].max() == 255 and _c[0, 6].mean() < 255
print("✓ Helpers ready (self-checks passed)")


✓ Helpers ready (self-checks passed)


In [4]:
# ── Cell 4: E6's pairs, the counting questions, and the checks that need no model ──
pairs_df = pd.read_csv(TEST_PAIRS)
pairs_df["left_out_because"] = pairs_df["left_out_because"].fillna("")
assert len(pairs_df) == EXPECTED["pairs"], f"{len(pairs_df)} pairs in test_pairs.csv; E6 had {EXPECTED['pairs']}"
assert int(pairs_df["clean"].sum()) == EXPECTED["clean"], "test_pairs.csv isn't E6's final file (2,007 clean pairs)"
unreadable = pairs_df["left_out_because"].str.contains("image unreadable")
jobs_df = pairs_df[~unreadable].reset_index(drop=True)
assert len(jobs_df) == EXPECTED["readable"], f"{len(jobs_df)} readable pairs; E6 had {EXPECTED['readable']}"

with open(BENCH_JSON, encoding="utf-8") as f:
    bench = json.load(f)
bdc = [e for e in bench if e.get("task") == TASK]
for i, e in enumerate(bdc):                       # ids as in the 499A scorer, H2 and E5: subset name + index in benchmark order
    e["id"] = f"building_damage_counting_{i}"
optical = [e for e in bdc if e.get("post_image_type") == "Optical"]
assert len(optical) == EXPECTED["questions"], f"{len(optical)} optical counting questions; expected {EXPECTED['questions']}"
with open(FLAGGED, encoding="utf-8") as f:
    flagged = list(csv.DictReader(f))
E5_COPIED = {r["id"] for r in flagged if r["task"] == "BDC" and r["sensor"] == "Optical" and r["flag"] == "after-image copied"}
assert len(E5_COPIED) == EXPECTED["e5_copied"], f"{len(E5_COPIED)} copied questions; E5 had {EXPECTED['e5_copied']}"

rows = []
for e in optical:
    opts, right = options_of(e), e["ground_truth_option"].strip()[0]
    name = name_of(e["post_image_path"])
    rows.append({"id": e["id"], "image": name, "event": event_of(name), "class": CLASS_OF[e["cls_description"]],
                 "truth": float(e["ground_truth"]), "truth_option": right, "options": opts, "largest_option": max(opts),
                 "e5_clean": e["id"] not in E5_COPIED, "no_image_rule | correct": no_image_rule(opts) == right,
                 "always_middle | correct": middle_option(opts) == right})
QDF = pd.DataFrame(rows)
assert set(QDF["image"]) == set(pairs_df["after_image"]), "the questions and E6's pairs are on different images"
rule_acc, middle_acc = round(100 * QDF["no_image_rule | correct"].mean(), 2), round(100 * QDF["always_middle | correct"].mean(), 2)
assert rule_acc == EXPECTED["rule_%"] and middle_acc == EXPECTED["middle_%"], f"The no-image rules give {rule_acc}% and {middle_acc}%"

bench_index = {p.name: p for p in BENCH_IMAGES.rglob("*.png")}
missing = [n for n in list(jobs_df["after_image"]) + list(jobs_df["before_image"]) if n not in bench_index]
if not SMOKE_TEST:
    assert not missing, f"{len(missing)} images aren't in {BENCH_IMAGES}, e.g. {missing[:3]}"

with open(MANIFEST_PATH, encoding="utf-8") as f:
    manifest = json.load(f)

def resolve_val(rel_path):
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for c in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path, DATA_ROOT / "train_images" / "train_images" / filename):
        if c.exists():
            return c
    return None

VAL_PATHS = {name_of(e["post_image_path"]): (e["post_image_path"], e["pre_image_path"]) for e in manifest
             if e.get("task") == "Referring Expression Segmentation" and e.get("image_type") == "Optical"
             and e.get("cls_description") == "Building Damage Assessment"}
del manifest
INST = {s: pd.read_csv(INST_PATHS[s]).query("found_by == 'sam3'") for s in SEEDS}
_names = sorted(n for n in INST[42]["image"].unique() if n in VAL_PATHS and resolve_val(VAL_PATHS[n][0]) and resolve_val(VAL_PATHS[n][1]))
_n = 3 if SMOKE_TEST else CHECK_IMAGES
CHECK_NAMES = _names[::max(len(_names) // _n, 1)][:_n]
assert CHECK_NAMES, "instances_val.csv and the manifest share no validation image"

if SMOKE_TEST:
    jobs_df = pd.concat([jobs_df[jobs_df["clean"]].head(10), jobs_df[~jobs_df["clean"]].head(2)]).reset_index(drop=True)
print(f"✓ {len(pairs_df):,} test pairs, {int(pairs_df['clean'].sum()):,} clean; {len(jobs_df):,} to run")
print(f"✓ {len(QDF):,} optical counting questions on those images, {int(QDF['e5_clean'].sum()):,} of them E5's clean ones")
print(f"✓ The rule that never sees the image scores {rule_acc}%, always the middle value {middle_acc}% (as checked on the PC)")
print(f"✓ Safety checks will use {len(CHECK_NAMES)} validation images, e.g. {CHECK_NAMES[0]}")


✓ 2,067 test pairs, 2,007 clean; 2,065 to run
✓ 3,583 optical counting questions on those images, 3,517 of them E5's clean ones
✓ The rule that never sees the image scores 66.98%, always the middle value 53.95% (as checked on the PC)
✓ Safety checks will use 12 validation images, e.g. bata_explosion_post_14.png


In [5]:
# ── Cell 5: SAM 3 on every before-image, in its own process, each map written as it comes ──
WORKER = r"""
import json, sys, time
from contextlib import nullcontext
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import sam3.model.vitdet as vitdet
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.model_builder import build_sam3_image_model

cfg = json.load(open(sys.argv[1]))
DEVICE = "cuda"
PRECISIONS = {"fp32": nullcontext, "fp16": lambda: torch.autocast("cuda", dtype=torch.float16)}

def _addmm_act_any_precision(activation, linear, mat1):
    # E3's fix: SAM 3's fused ViT-MLP op always computes in bfloat16, and a T4 has no bf16 tensor cores
    y = F.linear(mat1, linear.weight, linear.bias)
    return F.gelu(y) if activation in (F.gelu, torch.nn.GELU) else F.relu(y)

vitdet.addmm_act = _addmm_act_any_precision
t0 = time.time()
model = build_sam3_image_model(device=DEVICE)
processor = Sam3Processor(model, device=DEVICE, confidence_threshold=cfg["confidence"])
print(f"SAM 3 loaded in {time.time() - t0:.0f} s", flush=True)

def segment(image, grid, precision):
    # E3's segment(): the quarters separately, so each can return up to 200 objects; higher scores win any overlap
    h, w = image.shape[:2]
    ids = np.zeros((h, w), np.uint16)
    next_id = 1
    for gy in range(grid):
        for gx in range(grid):
            y0, y1, x0, x1 = gy * h // grid, (gy + 1) * h // grid, gx * w // grid, (gx + 1) * w // grid
            with PRECISIONS[precision]():
                state = processor.set_image(Image.fromarray(image[y0:y1, x0:x1]))
                out = processor.set_text_prompt(prompt=cfg["prompt"], state=state)
            masks = out["masks"][:, 0].cpu().numpy()
            scores = out["scores"].float().cpu().numpy()
            tile = ids[y0:y1, x0:x1]
            for k in np.argsort(scores):
                if masks[k].sum() >= cfg["min_instance_px"]:
                    tile[masks[k]] = next_id
                    next_id += 1
    return ids

t0, done, skipped = time.time(), 0, []
for path, out_path in cfg["jobs"]:
    image = cv2.imread(path)
    if image is None:
        skipped.append(path)
        continue
    ids = segment(cv2.cvtColor(image, cv2.COLOR_BGR2RGB), cfg["grid"], cfg["precision"])
    cv2.imwrite(out_path, ids)
    done += 1
    if done % 100 == 0:
        print(f"  {done:,}/{len(cfg['jobs']):,} maps, {(time.time() - t0) / done:.2f} s each", flush=True)
print(f"DONE {done} maps in {(time.time() - t0) / 60:.1f} min; skipped {skipped}", flush=True)
"""

SAM_SCRIPT, SAM_JOBS = Path("/kaggle/temp/sam_worker.py"), Path("/kaggle/temp/sam_jobs.json")
SAM_SCRIPT.write_text(WORKER, encoding="utf-8")
sam_jobs = [(str(resolve_val(VAL_PATHS[n][1])), str(CHECK_DIR / f"{Path(n).stem}.png")) for n in CHECK_NAMES]
sam_jobs += [(str(bench_index[r.before_image]), str(MAP_DIR / f"{Path(r.after_image).stem}.png"))
             for r in jobs_df.itertuples()]         # each map is named after its after-image, as in E3
SAM_JOBS.write_text(json.dumps({"jobs": sam_jobs, "prompt": PROMPT, "confidence": CONFIDENCE, "grid": GRID,
                                "min_instance_px": MIN_INSTANCE_PX, "precision": PRECISION}), encoding="utf-8")
t_sam = time.time()
process = subprocess.Popen([sys.executable, "-u", str(SAM_SCRIPT), str(SAM_JOBS)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, env=dict(os.environ, PYTHONPATH=SAM_ENV, HF_TOKEN=TOKEN, CUDA_VISIBLE_DEVICES="0"))
sam_log = []
for line in process.stdout:
    sam_log.append(line)
    print(line, end="")
if process.wait() != 0:
    (OUT_DIR / "sam_worker.log").write_text("".join(sam_log), encoding="utf-8")
    raise RuntimeError("SAM 3 stopped: its log is above and in sam_worker.log")
SAM_MIN = (time.time() - t_sam) / 60
n_maps = len(list(MAP_DIR.glob("*.png")))
assert n_maps == len(jobs_df), f"{n_maps} SAM 3 maps for {len(jobs_df)} test pairs"
print(f"✓ SAM 3: {n_maps:,} test maps in {SAM_MIN:.1f} min")


/kaggle/temp/sam3_env/sam3/model_builder.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/kaggle/temp/sam3_env/sam3/model/model_misc.py:71: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()
/kaggle/temp/sam3_env/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
SAM 3 loaded in 40 s
  100/2,077 maps, 2.49 s each
  200/2,077 maps, 2.50 s each
  300/2,077 maps, 2.50 s each
  400/2,077 maps, 2.51 s each
  500/2,077 maps, 2.51 s each
  600/2,077 maps, 2.50 s each
  700/2,077 maps, 

In [6]:
# ── Cell 6: The graders and R11, then the safety checks on validation images ──
import segmentation_models_pytorch as smp
import torch.nn as nn
import torch.nn.functional as F
import torchvision

class SiameseGrader(nn.Module):
    """A shared ResNet-18 reads the before and the after crop (RGB + building mask each);
    [f_before, f_after, f_after - f_before] -> one score per label. R16c step 5's model."""

    def __init__(self):
        super().__init__()
        net = torchvision.models.resnet18(weights=None)
        net.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
        net.fc = nn.Identity()
        self.encoder = net
        self.head = nn.Sequential(nn.Linear(3 * 512, 256), nn.ReLU(inplace=True), nn.Dropout(0.2), nn.Linear(256, len(LABELS)))

    def forward(self, before, after):
        fb, fa = self.encoder(before), self.encoder(after)
        return self.head(torch.cat([fb, fa, fa - fb], 1))

MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1) * 255
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1) * 255

def to_inputs(x):
    """uint8 (B, 7, S, S) -> normalised before and after tensors (B, 4, S, S); the mask becomes -1 / +1."""
    x = x.to(DEVICE, non_blocking=True).float()
    mask = x[:, 6:7] / 127.5 - 1
    return torch.cat([(x[:, 0:3] - MEAN) / STD, mask], 1), torch.cat([(x[:, 3:6] - MEAN) / STD, mask], 1)

@torch.no_grad()
def log_probs(model, crops):
    """uint8 crops (N, 7, S, S) -> log-probabilities (N, labels)."""
    model.eval()
    out = []
    for i in range(0, len(crops), 512):
        before, after = to_inputs(torch.from_numpy(np.ascontiguousarray(crops[i:i + 512])))
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            out.append(torch.log_softmax(model(before, after).float(), 1).cpu())
    return torch.cat(out).numpy() if out else np.zeros((0, len(LABELS)), np.float32)

def load_grader(path):
    model = SiameseGrader()
    model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    return model.to(DEVICE).eval()

def load_unet(path):
    """D4 checkpoint: U-Net + ResNet-50, 6 input channels (after-image, then before-image), 4 classes."""
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=6, classes=NUM_CLASSES)
    model.load_state_dict(state)
    return model.to(DEVICE).eval()

IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
IMNET_STD = np.array([0.229, 0.224, 0.225], np.float32) * 255

def unet_input(post, pre):
    """As R11's val_transform: ImageNet normalisation, channels 0-2 after-image, 3-5 before-image."""
    x = np.concatenate([(post.astype(np.float32) - IMNET_MEAN) / IMNET_STD,
                        (pre.astype(np.float32) - IMNET_MEAN) / IMNET_STD], 2)
    return torch.from_numpy(x.transpose(2, 0, 1).copy()).unsqueeze(0).to(DEVICE)

def _forward_once(model, x):
    """Logits at x's own HxW: pad to a multiple of 32, exactly as in the D4 notebooks."""
    h, w = x.shape[-2:]
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    if ph or pw:
        x = F.pad(x, (0, pw, 0, ph), mode="reflect")
    return model(x)[..., :h, :w].float()

TTA_VIEWS = [(k, flip) for flip in (False, True) for k in range(4)]   # the D4 notebooks' 8 views

def _tta_fwd(x, k, flip):
    x = torch.rot90(x, k, dims=(2, 3))
    return torch.flip(x, dims=(3,)) if flip else x

def _tta_inv(y, k, flip):
    if flip:
        y = torch.flip(y, dims=(3,))
    return torch.rot90(y, -k, dims=(2, 3))

@torch.no_grad()
def predict_std_and_tta(model, x):
    """One pass over the 8 views (as in E1). The identity view (k=0, no flip) is the standard prediction."""
    prob_sum, pred_std = 0, None
    with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
        for k, flip in TTA_VIEWS:
            logits = _forward_once(model, _tta_fwd(x, k, flip))
            if k == 0 and not flip:
                pred_std = logits.argmax(1)
            prob_sum = prob_sum + torch.softmax(_tta_inv(logits, k, flip), dim=1)
    return pred_std[0].cpu().numpy().astype(np.uint8), (prob_sum / len(TTA_VIEWS)).argmax(1)[0].cpu().numpy().astype(np.uint8)

graders = {s: load_grader(GRADER_PATHS[s]) for s in SEEDS}
unet = load_unet(UNET_CKPT)
print(f"✓ Graders {list(graders)} and {UNET} loaded")

R11_COLS = ["r11_background_px", "r11_intact_px", "r11_damaged_px", "r11_destroyed_px"]

def safety_checks():
    """Each model must reproduce what it saved on these validation images. Returns one row per image."""
    rows = []
    for name in CHECK_NAMES:
        stem = Path(name).stem
        post_rel, pre_rel = VAL_PATHS[name]
        post, pre = read_pair(resolve_val(post_rel), resolve_val(pre_rel))
        shape = post.shape[:2]
        old, new = read_sam(E3_DIR / f"{stem}.png", shape), read_sam(CHECK_DIR / f"{stem}.png", shape)
        union = int(((old > 0) | (new > 0)).sum())
        row = {"image": name, "sam_iou": round(int(((old > 0) & (new > 0)).sum()) / max(union, 1), 4)}
        worst = 0.0
        for seed in SEEDS:
            saved = INST[seed][INST[seed]["image"] == name]
            ids = saved["id"].tolist()
            mine = log_probs(graders[seed], building_crops(pre, post, old, ids))
            theirs = saved[[f"logp_{l.lower()}_s{seed}" for l in LABELS]].to_numpy(np.float64)
            worst = max(worst, float(np.abs(mine - theirs).max()) if len(ids) else 0.0)
        row["logp_max_difference"] = round(worst, 4)
        _, tta = predict_std_and_tta(unet, unet_input(post, pre))
        saved = INST[SEEDS[0]][INST[SEEDS[0]]["image"] == name]
        size = int(old.max()) + 1
        px = np.bincount(old.ravel().astype(np.int64) * NUM_CLASSES + tta.ravel(), minlength=size * NUM_CLASSES).reshape(size, NUM_CLASSES)
        ids = saved["id"].to_numpy()
        different = int(np.abs(px[ids] - saved[R11_COLS].to_numpy()).sum()) if len(ids) else 0
        row["r11_px_share_different"] = round(different / max(int(px[ids].sum()), 1), 4)
        rows.append(row)
    return pd.DataFrame(rows)

def judge_safety(df):
    """Stops the notebook if a model doesn't reproduce what it saved."""
    failures = []
    if df["sam_iou"].mean() < LIMITS["sam_iou"]:
        failures.append(f"SAM 3 differs from E3's maps: mean building IoU {df['sam_iou'].mean():.4f}, limit {LIMITS['sam_iou']}")
    if df["logp_max_difference"].max() > LIMITS["logp"]:
        failures.append(f"A grader differs from its saved log-probabilities by {df['logp_max_difference'].max()}, limit {LIMITS['logp']}")
    if df["r11_px_share_different"].max() > LIMITS["r11_px"]:
        failures.append(f"R11 differs from its saved pixels by {df['r11_px_share_different'].max()}, limit {LIMITS['r11_px']}")
    if failures:
        raise RuntimeError("Safety check failed, so the long run didn't start: " + "; ".join(failures))
    return True


✓ Graders [42, 43, 44] and R11 loaded


In [7]:
# ── Cell 7: Run the safety checks ───────────────────────────────────────────
SAFETY = safety_checks()
print(SAFETY.to_string(index=False))
SAFETY.to_csv(OUT_DIR / "safety_checks.csv", index=False)
judge_safety(SAFETY)
print(f"✓ On {len(SAFETY)} validation images: SAM 3 reproduces E3's maps (mean building IoU {SAFETY['sam_iou'].mean():.4f}), "
      f"the graders their log-probabilities (largest difference {SAFETY['logp_max_difference'].max()}), "
      f"R11 its pixels (largest share different {SAFETY['r11_px_share_different'].max()})")


                                        image  sam_iou  logp_max_difference  r11_px_share_different
                   bata_explosion_post_14.png   1.0000               0.0078                  0.0000
hurricane_florence_00000443_post_disaster.png   0.9999               0.0043                  0.0001
  hurricane_harvey_00000451_post_disaster.png   0.9986               0.0101                  0.0001
 hurricane_michael_00000407_post_disaster.png   0.9999               0.0117                  0.0000
  midwest_flooding_00000287_post_disaster.png   0.9997               0.0041                  0.0000
    nepal_flooding_00000339_post_disaster.png   1.0000               0.0048                  0.0000
   pinery_bushfire_00001507_post_disaster.png   0.9995               0.0047                  0.0005
 portugal_wildfire_00000758_post_disaster.png   0.9999               0.0077                  0.0000
 portugal_wildfire_00001764_post_disaster.png   0.9997               0.0046                  0.0000


In [8]:
# ── Cell 8: The chain and R11 on every test pair ────────────────────────────
METHODS = [f"chain_s{s}" for s in SEEDS] + ["R11_tta", "R11_std", "ceiling_sam3"]   # ceiling: a perfect grader on SAM 3's outlines
cms = {m: defaultdict(lambda: np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)) for m in METHODS}
true_cm = {s: np.zeros((3, len(LABELS)), np.int64) for s in SEEDS}     # rows: true grade, columns: label
counts, inst_frames, not_run = {}, [], []
DB_PATH = OUT_DIR / f"buildings_chain_s{TABLE_SEED}.sqlite"
DB_PATH.unlink(missing_ok=True)
conn = sqlite3.connect(DB_PATH)
create_schema(conn)
t_run = time.time()
for r in tqdm(list(jobs_df.itertuples()), desc="Test pairs"):
    name = r.after_image
    data = read_pair(bench_index[name], bench_index[r.before_image])
    if data is None:
        not_run.append(name)
        continue
    post, pre = data
    h, w = post.shape[:2]
    stem, event = Path(name).stem, event_of(name)
    ids = read_sam(MAP_DIR / f"{stem}.png", (h, w))
    pred_std, pred_tta = predict_std_and_tta(unet, unet_input(post, pre))

    present = np.unique(ids)
    present = present[present > 0]
    size = int(ids.max()) + 1
    crops = building_crops(pre, post, ids, present) if len(present) else None
    logps, preds = {}, {"R11_tta": pred_tta, "R11_std": pred_std}
    for seed in SEEDS:
        lp = log_probs(graders[seed], crops) if len(present) else np.zeros((0, len(LABELS)), np.float32)
        logps[seed] = lp
        lut = np.zeros(size, np.uint8)
        lut[present] = to_class((lp + BIASES[seed]).argmax(1))
        preds[f"chain_s{seed}"] = lut[ids]
        counts[(name, f"chain_s{seed}")] = blob_counts(preds[f"chain_s{seed}"])
    counts[(name, "R11")] = blob_counts(pred_tta)
    table_map = preds[f"chain_s{TABLE_SEED}"]
    insert_image(conn, name, h, w, blobs_of(table_map))
    cv2.imwrite(str(CHAIN_DIR / name), table_map)
    cv2.imwrite(str(R11_DIR / name), pred_tta)

    flat = ids.ravel().astype(np.int64) * NUM_CLASSES
    r11_px = np.bincount(flat + pred_tta.ravel(), minlength=size * NUM_CLASSES).reshape(size, NUM_CLASSES)
    frame = {"image": name, "event": event, "id": present, "area_px": r11_px[present].sum(1)}
    for seed in SEEDS:
        for j, l in enumerate(LABELS):
            frame[f"logp_{l.lower()}_s{seed}"] = np.round(logps[seed][:, j], 6) if len(present) else []
    for i, c in enumerate(CLASS_NAMES):
        frame[f"r11_{c.lower()}_px"] = r11_px[present, i]

    if r.clean:                              # scored only on E6's clean pairs
        gt = read_mask(name, (h, w))
        gt_px = np.bincount(flat + gt.ravel(), minlength=size * NUM_CLASSES).reshape(size, NUM_CLASSES)
        perfect = np.zeros(size, np.uint8)
        perfect[present] = gt_px[present, 1:].argmax(1) + 1      # each outline's main true grade
        preds["ceiling_sam3"] = perfect[ids]
        for m, p in preds.items():
            cms[m][event] += confusion(gt, p)
        for i, c in enumerate(CLASS_NAMES):
            frame[f"gt_{c.lower()}_px"] = gt_px[present, i]
        parts = true_pieces(gt)
        if parts:
            true_crops = np.concatenate([building_crops(pre, post, lab, part_ids) for _, lab, part_ids in parts])
            true_labels = np.concatenate([np.full(len(part_ids), l) for l, _, part_ids in parts])
            for seed in SEEDS:
                np.add.at(true_cm[seed], (true_labels, (log_probs(graders[seed], true_crops) + BIASES[seed]).argmax(1)), 1)
    if len(present):
        inst_frames.append(pd.DataFrame(frame))
conn.commit()
n_rows = conn.execute("SELECT COUNT(*) FROM buildings").fetchone()[0]
by_class = dict(conn.execute("SELECT class_name, COUNT(*) FROM buildings GROUP BY class_name").fetchall())
conn.close()
RUN_MIN = (time.time() - t_run) / 60
assert len(not_run) <= 0.01 * len(jobs_df), f"{len(not_run)} pairs couldn't be read"
pd.concat(inst_frames, ignore_index=True).to_csv(OUT_DIR / "instances_test.csv", index=False)
print(f"✓ {len(jobs_df) - len(not_run):,} test pairs in {RUN_MIN:.1f} min; table: {n_rows:,} buildings {by_class}")
if not_run:
    print(f"⚠ Not run ({len(not_run)}): {not_run[:5]}")


Test pairs:   0%|          | 0/2065 [00:00<?, ?it/s]

✓ 2,065 test pairs in 37.3 min; table: 104,596 buildings {'damaged': 9968, 'destroyed': 7629, 'intact': 86999}


In [9]:
# ── Cell 9: Map scores on E6's clean pairs, per grader, as the mean of three, per event ──
def merged(method, keep=lambda event: True):
    total = np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)
    for event, cm in cms[method].items():
        if keep(event):
            total += cm
    return total

CHAIN = [f"chain_s{s}" for s in SEEDS]
SCOPES = {"clean pairs": lambda e: True, "clean pairs, without Turkey": lambda e: not e.startswith("turkey"),
          "Turkey only": lambda e: e.startswith("turkey")}
rows = []
for scope, keep in SCOPES.items():
    scored = {m: metrics(merged(m, keep)) for m in METHODS}
    for m in METHODS:
        rows.append({"scope": scope, "method": m, **scored[m]})
    rows.append({"scope": scope, "method": "chain, mean of three",
                 **{k: round(float(np.mean([scored[m][k] for m in CHAIN])), 4) for k in scored[CHAIN[0]]}})
map_scores = pd.DataFrame(rows)
map_scores.to_csv(OUT_DIR / "map_scores.csv", index=False)
print(map_scores[["scope", "method", "miou", "intact_iou", "damaged_iou", "destroyed_iou", "building_iou", "destroyed_precision",
                  "destroyed_recall"]].to_string(index=False))

n_by_event = jobs_df[jobs_df["clean"]].assign(event=lambda d: d["after_image"].map(event_of)).groupby("event").size().to_dict()
event_rows = []
for event in sorted(n_by_event, key=lambda e: -n_by_event[e]):
    scored = {m: metrics(cms[m][event]) for m in METHODS}
    event_rows.append({"event": event, "clean_pairs": n_by_event[event],
                       "chain_destroyed_iou_mean_of_three": round(float(np.mean([scored[m]["destroyed_iou"] for m in CHAIN])), 4),
                       "r11_tta_destroyed_iou": scored["R11_tta"]["destroyed_iou"],
                       "chain_miou_mean_of_three": round(float(np.mean([scored[m]["miou"] for m in CHAIN])), 4),
                       "r11_tta_miou": scored["R11_tta"]["miou"],
                       "ceiling_destroyed_iou": scored["ceiling_sam3"]["destroyed_iou"],
                       "true_destroyed_px": int(merged("R11_tta", lambda e: e == event)[3].sum())})
by_event = pd.DataFrame(event_rows)
by_event.to_csv(OUT_DIR / "map_scores_by_event.csv", index=False)
print("\nPer event (an event with no true destroyed pixels scores 0 for every method):")
print(by_event.to_string(index=False))

true_rows, macro_f1 = [], {}
for seed in SEEDS:
    cm = true_cm[seed].astype(np.float64)
    tp = np.diag(cm[:, :3])
    precision = tp / np.maximum(cm[:, :3].sum(0), 1)
    recall = tp / np.maximum(cm.sum(1), 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-8)
    macro_f1[seed] = round(float(f1.mean()), 4)
    true_rows += [{"seed": seed, "grade": g, "buildings": int(cm[i].sum()), "precision": round(float(precision[i]), 4),
                   "recall": round(float(recall[i]), 4), "f1": round(float(f1[i]), 4),
                   **{f"graded_{l.lower()}": int(true_cm[seed][i, j]) for j, l in enumerate(LABELS)}}
                  for i, g in enumerate(GRADES)]
true_df = pd.DataFrame(true_rows)
true_df.to_csv(OUT_DIR / "grader_true_buildings.csv", index=False)
print(f"\nThe graders on the true buildings of the clean pairs, macro F1: {macro_f1} (validation: 0.7367, 0.7303, 0.754)")


                      scope               method   miou  intact_iou  damaged_iou  destroyed_iou  building_iou  destroyed_precision  destroyed_recall
                clean pairs            chain_s42 0.5179      0.4888       0.2513         0.4055        0.5395               0.6387            0.5262
                clean pairs            chain_s43 0.5121      0.4778       0.2460         0.3983        0.5405               0.6416            0.5123
                clean pairs            chain_s44 0.5087      0.4908       0.2516         0.3657        0.5447               0.5091            0.5650
                clean pairs              R11_tta 0.5721      0.6407       0.3057         0.3984        0.6944               0.5631            0.5766
                clean pairs              R11_std 0.5613      0.6320       0.2869         0.3848        0.6869               0.5395            0.5730
                clean pairs         ceiling_sam3 0.6648      0.5341       0.6092         0.5898        0.5

In [10]:
# ── Cell 10: Count scores: accuracy, count error and the rule that never sees the image ──
for m in CHAIN + ["R11"]:
    QDF[f"{m} | count"] = [counts[(im, m)][c - 1] if (im, m) in counts else np.nan for im, c in zip(QDF["image"], QDF["class"])]
    QDF[f"{m} | correct"] = [nearest_option(n, o) == t if n == n else False
                             for n, o, t in zip(QDF[f"{m} | count"], QDF["options"], QDF["truth_option"])]
ANSWERED = QDF[QDF["R11 | count"].notna()].copy()          # the three questions on the two unreadable pairs can't be asked
ANSWERED["class_name"] = ANSWERED["class"].map(CLASS_NAME)

def accuracy(df, m):
    return round(100 * df[f"{m} | correct"].mean(), 2) if len(df) else float("nan")

def count_error(df, m):
    err = (df[f"{m} | count"] - df["truth"]).abs()
    pct = err / df["truth"].clip(lower=1)
    return {"mean_abs_error": round(float(err.mean()), 2), "median_abs_error": float(err.median()),
            "median_error_%": round(100 * float(pct.median()), 1), "within_20%_%": round(100 * float((pct <= 0.2).mean()), 1)}

clean_q = ANSWERED[ANSWERED["e5_clean"]]
big = ANSWERED["largest_option"] > 20
rows = []
for m in CHAIN + ["R11", "no_image_rule", "always_middle"]:
    row = {"method": m, "all_%": accuracy(ANSWERED, m), "e5_clean_%": accuracy(clean_q, m),
           **{f"{c}_%": accuracy(ANSWERED[ANSWERED["class_name"] == c], m) for c in CLASS_NAME.values()},
           "largest_above_20_%": accuracy(ANSWERED[big], m), "largest_up_to_20_%": accuracy(ANSWERED[~big], m)}
    if f"{m} | count" in ANSWERED:
        row.update(count_error(ANSWERED, m))
    rows.append(row)
counting = pd.DataFrame(rows)
counting.to_csv(OUT_DIR / "counting_scores.csv", index=False)
print(f"{len(ANSWERED):,} questions answered ({len(clean_q):,} of them E5's clean ones). For scale: base Qwen2.5-VL-7B "
      f"{EXPECTED['base_%']}% on the optical questions (paper).")
print(counting.to_string(index=False))

TABLE = f"chain_s{TABLE_SEED}"
paired = {"E5's clean questions": mcnemar(clean_q[f"{TABLE} | correct"], clean_q["R11 | correct"]),
          "all optical questions": mcnemar(ANSWERED[f"{TABLE} | correct"], ANSWERED["R11 | correct"])}
print(f"\nSeed {TABLE_SEED}'s table against R11's, same questions (exact McNemar): {paired}")
closer = (ANSWERED[f"{TABLE} | count"] - ANSWERED["truth"]).abs() - (ANSWERED["R11 | count"] - ANSWERED["truth"]).abs()
closer_counts = {"chain_closer": int((closer < 0).sum()), "same": int((closer == 0).sum()), "r11_closer": int((closer > 0).sum())}
print(f"Which count is closer to the truth, question by question: {closer_counts}")
QDF.drop(columns="options").to_csv(OUT_DIR / "per_question.csv", index=False)

# R11's table must reproduce H2's numbers: H2 ran the same model on the same questions
r11_all, r11_clean = accuracy(ANSWERED, "R11"), accuracy(clean_q, "R11")
R11_OK = (abs(r11_all - EXPECTED["r11_all_%"]) <= LIMITS["r11_points"] and abs(r11_clean - EXPECTED["r11_clean_%"]) <= LIMITS["r11_points"])
if SMOKE_TEST:
    r11_status = "n/a (smoke test)"
else:
    r11_status = "yes" if R11_OK else "NO"
print(f"R11's table reproduces H2's {EXPECTED['r11_all_%']}% and E5's {EXPECTED['r11_clean_%']}%: {r11_status} ({r11_all}%, {r11_clean}%)")
if r11_status == "NO":
    print("⚠ R11 isn't the model H2 ran, or the images differ. Check this before trusting the comparison.")


3,580 questions answered (3,514 of them E5's clean ones). For scale: base Qwen2.5-VL-7B 34.2% on the optical questions (paper).
       method  all_%  e5_clean_%  intact_%  damaged_%  destroyed_%  largest_above_20_%  largest_up_to_20_%  mean_abs_error  median_abs_error  median_error_%  within_20%_%
    chain_s42  32.88       32.70     31.20      32.07        38.27               23.54               40.25           13.58               4.0            40.0          30.7
    chain_s43  30.95       30.85     28.65      31.55        36.08               21.71               38.25           14.07               5.0            41.9          30.3
    chain_s44  34.16       33.95     32.96      32.79        39.09               25.76               40.80           13.08               4.0            36.7          33.3
          R11  33.02       32.53     29.29      35.46        39.37               22.22               41.55           13.34               4.0            38.0          32.3
no_image_rule  66

In [11]:
# ── Cell 11: The rule fixed before the run, the expectation, and the output files ──
def score(method, scope="clean pairs", key="destroyed_iou"):
    return map_scores[(map_scores["scope"] == scope) & (map_scores["method"] == method)][key].item()

chain_d, r11_d = score("chain, mean of three"), score("R11_tta")
chain_m, r11_m = score("chain, mean of three", key="miou"), score("R11_tta", key="miou")
chain_acc, r11_acc = accuracy(clean_q, TABLE), accuracy(clean_q, "R11")
P_CLEAN = paired["E5's clean questions"]["p"]
OUTCOME, SENTENCE = verdict(chain_d, r11_d, chain_acc, r11_acc, P_CLEAN)
if SMOKE_TEST:
    OUTCOME, SENTENCE = "smoke test", "Smoke test: no verdict."
expect = {"chain lower on test than on clean validation (0.3065)": chain_d < EXPECTED["val_chain_destroyed"],
          "R11 lower on test than on clean validation (0.2692)": r11_d < EXPECTED["val_r11_destroyed"],
          "counting about level with R11's (p of at least 0.05)": P_CLEAN >= 0.05}
print(f"Destroyed-building IoU on the clean test pairs: chain (mean of three) {chain_d}, R11 with TTA {r11_d}. mIoU: {chain_m}, {r11_m}")
print(f"Counting on E5's clean questions: seed {TABLE_SEED}'s table {chain_acc}%, R11's {r11_acc}%, p = {P_CLEAN}")
print(f"\n{SENTENCE}")
for k, v in ({} if SMOKE_TEST else expect).items():
    print(f"  Expected: {k}: {'yes' if v else 'NO'}")

for folder, out_name in ((CHAIN_DIR, f"chain_s{TABLE_SEED}_maps"), (R11_DIR, "r11_tta_maps"), (MAP_DIR, "sam3_pre_grid2")):
    shutil.make_archive(str(OUT_DIR / out_name), "zip", folder)

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: a plain-text table instead
        return "```\n" + df.to_string(index=False) + "\n```"

config = {"task": "E7 step 2", "smoke_test": SMOKE_TEST, "sam3_commit": SAM3_COMMIT, "prompt": PROMPT, "confidence": CONFIDENCE, "grid": GRID,
          "precision": PRECISION, "seeds": list(SEEDS), "table_seed": TABLE_SEED, "biases": {str(s): BIASES[s].tolist() for s in SEEDS},
          "unet_checkpoint": str(UNET_CKPT), "test_pairs_run": len(jobs_df) - len(not_run), "not_run": not_run, "questions_answered": len(ANSWERED),
          "table_buildings": n_rows, "table_by_class": by_class, "sam_minutes": round(SAM_MIN, 1), "run_minutes": round(RUN_MIN, 1),
          "outcome": OUTCOME, "paired": paired, "closer_to_truth": closer_counts, "macro_f1_true_buildings": {str(k): v for k, v in macro_f1.items()},
          "r11_reproduces_h2": r11_status, "expectation": {k: bool(v) for k, v in expect.items()}}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
SCOPE_COLS = ["method", "miou", "intact_iou", "damaged_iou", "destroyed_iou", "building_iou", "destroyed_precision", "destroyed_recall"]
tables = [line for scope in SCOPES for line in (f"### {scope}", "", as_markdown(map_scores[map_scores["scope"] == scope][SCOPE_COLS]), "")]
(OUT_DIR / "README.md").write_text("\n".join([
    "# E7 step 2: the whole chain on the unseen test pairs", "",
    ("**SMOKE TEST: checks the code, not the answer.** " if SMOKE_TEST else "") +
    f"Run on {len(jobs_df) - len(not_run):,} optical test pairs; maps scored on E6's {int(jobs_df['clean'].sum()):,} clean pairs.", "",
    "- **The chain:** SAM 3 (prompt \"building\", on the before-photo in four quarters) finds each building. A Siamese ResNet-18 trained on SAM 3's "
    "outlines of the training images labels each one intact, damaged, destroyed or not a building, from 128 px before and after crops.",
    "- **The baseline, R11:** a U-Net with a ResNet-50 encoder, with the after-photo and before-photo as 6 input channels, using 8-view test-time "
    "augmentation (TTA).", "",
    "## The result, by the rule fixed before the run", "",
    f"**{SENTENCE}**", "",
    f"- Destroyed-building IoU on the clean test pairs: chain (mean of three graders) **{chain_d}**, R11 with TTA **{r11_d}**.",
    f"- Mean IoU: chain {chain_m}, R11 with TTA {r11_m}.",
    f"- Counting on E5's clean questions: seed {TABLE_SEED}'s table {chain_acc}%, R11's {r11_acc}%, exact McNemar p = {P_CLEAN}.",
    "- O1 stays judged on validation, as the review set it. Nothing is tuned on these pairs.", "",
    "## Map scores", "", *tables,
    "`ceiling_sam3`: a perfect grader on SAM 3's outlines, the most a better grader could reach on these outlines.", "",
    "## Per event", "", as_markdown(by_event), "",
    "## The graders on the true buildings (clean pairs)", "",
    f"Macro F1: {macro_f1}. On validation: 0.7367, 0.7303, 0.754.", "", as_markdown(true_df), "",
    "## Counting", "",
    f"{len(ANSWERED):,} of {len(QDF):,} optical counting questions could be asked: the others are on the two pairs with an unreadable image. Each method "
    "counts 8-connected blobs of at least 20 px and picks the nearest option.", "", as_markdown(counting), "",
    "- **Read accuracy with care.** `no_image_rule` and `always_middle` never see the image. When a question's largest option is above 20, the options are "
    "built around the correct count, so the middle value is always right. Accuracy on these options, and the 34.2% bar, can't show that a method counts well.",
    "- **Count error** is against each question's true count: `median_error_%` is the median of the error as a share of the truth, and `within_20%_%` the "
    "share of counts within 20% of it.",
    f"- Which count is closer to the truth, question by question: {closer_counts}.", "",
    "Seed {}'s table against R11's on the same questions (exact McNemar):".format(TABLE_SEED), "",
    *[f"- {k}: {v}" for k, v in paired.items()], "",
    f"R11's table reproduces H2's {EXPECTED['r11_all_%']}% and E5's {EXPECTED['r11_clean_%']}%: **{r11_status}** ({r11_all}%, {r11_clean}%).", "",
    "## Against the expectation, fixed before the run", "",
    *[f"- {k}: **{'yes' if v else 'NO'}**" for k, v in expect.items()], "",
    "## Safety checks on validation images", "", as_markdown(SAFETY), "",
    "## Files", "",
    "- `map_scores.csv`, `map_scores_by_event.csv`, `grader_true_buildings.csv`, `counting_scores.csv`, `per_question.csv`, `safety_checks.csv`, `config.json`",
    "- `instances_test.csv`: each found building, its three graders' log-probabilities, R11's pixels and, on clean pairs, the true pixels",
    f"- `buildings_chain_s{TABLE_SEED}.sqlite`: the table, in H2's columns",
    f"- `chain_s{TABLE_SEED}_maps.zip`, `r11_tta_maps.zip`, `sam3_pre_grid2.zip`: for step 3 (hot spots, on the PC)",
]) + "\n", encoding="utf-8")
print(f"\n✓ Outputs in {OUT_DIR}: " + ", ".join(sorted(p.name for p in OUT_DIR.iterdir())))


Destroyed-building IoU on the clean test pairs: chain (mean of three) 0.3898, R11 with TTA 0.3984. mIoU: 0.5129, 0.5721
Counting on E5's clean questions: seed 42's table 32.7%, R11's 32.53%, p = 0.8743

The chain's destroyed-building IoU isn't above R11's: the lead doesn't hold on unseen data, and the choice of chain reopens before T2 step 5.
  Expected: chain lower on test than on clean validation (0.3065): NO
  Expected: R11 lower on test than on clean validation (0.2692): NO
  Expected: counting about level with R11's (p of at least 0.05): yes

✓ Outputs in /kaggle/working/step2: README.md, buildings_chain_s42.sqlite, chain_s42_maps.zip, config.json, counting_scores.csv, grader_true_buildings.csv, instances_test.csv, map_scores.csv, map_scores_by_event.csv, per_question.csv, r11_tta_maps.zip, safety_checks.csv, sam3_pre_grid2.zip
